# sparse_knn on a commercial model — gpt-5.6-sol, no thinking

---## 1 — Host and working treeSelection runs locally on the embedding index, so a GPU runtime only shortens the indexbuild. Generation is API-only.

In [ ]:
from pathlib import Pathif not Path('manage.py').exists():    if not Path('Style-Aware-MT/manage.py').exists():        !git clone https://github.com/prnamhr/Style-Aware-MT.git    %cd Style-Aware-MT!git pull --ff-only!git rev-parse --short HEAD

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
# The pipeline is text-only and these three ship against a torch the pins contradict.%pip uninstall -q -y torchvision torchaudio torchcodec

In [ ]:
import getpassimport loggingimport osif not os.environ.get('OPENAI_API_KEY'):    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY: ')logging.getLogger('httpx').setLevel(logging.WARNING)print('OPENAI_API_KEY set')

---## 2 — Run parameters and the matched-contrast gate

In [ ]:
import hashlibimport jsonimport subprocessimport sysimport tempfileimport timefrom datetime import datetime, timezonefrom pathlib import Pathimport numpy as npimport yamlPY = sys.executableCOMET_PY = '.venv-comet/bin/python'SPLIT = 'val'ARM, BASELINE = 'sparse_knn', 'knn_fewshot'ARM_NAME, BASE_NAME = 'gpt56_sparse_knn', 'gpt56_knn_fewshot'CONFIG = Path('configs/commercial_gpt56_sparse_knn.yaml')QWEN_CONFIG, BASE_CONFIG = Path('configs/sparse_knn.yaml'), Path('configs/base_qwen.yaml')OUT = Path('outputs')DIAG = Path(f'results/sparse_selection_{SPLIT}.json')N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42CFG = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))GEN, RETR, SPA = CFG['generator'], CFG['retrieval'], CFG['sparse']RAR, PROMPT = CFG['rarity'], CFG['prompt']RARITY_PATH = Path(RAR['out'])print(f"{GEN['model']}: {ARM} against {BASELINE}, k={RETR['k']}, up to m={SPA['m']} rare, "      f"{RAR['freeze_n']} rarest terms at df >= {RAR['min_df']}")

In [ ]:
QWEN = yaml.safe_load(QWEN_CONFIG.read_text(encoding='utf-8'))for block in ('prompt', 'retrieval', 'rarity', 'sparse', 'data'):    assert CFG[block] == QWEN[block], f'{block} differs from {QWEN_CONFIG}: {CFG[block]}'assert set(CFG) == set(QWEN), sorted(set(CFG) ^ set(QWEN))assert RETR == yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))['retrieval'], RETRprint(f'{CONFIG.name} differs from {QWEN_CONFIG.name} in the generator and output name only')

In [ ]:
# A thinking budget would let deliberation, not selection, carry the contrast.assert GEN['reasoning_effort'] == 'none', GEN['reasoning_effort']assert GEN['temperature'] is None, 'gpt-5.x rejects a custom temperature; leave it null'assert GEN['seed'] == 42 and GEN['pricing'], GENassert CFG['data']['eval_file'].endswith(f'{SPLIT}.jsonl'), 'test split stays sealed'assert CFG['data']['limit'] is None, 'limit must be null for the full pass'assert {ARM_NAME, BASE_NAME}.isdisjoint({ARM, BASELINE}), 'names collide with the Qwen outputs'print(f"{GEN['model']} at reasoning_effort={GEN['reasoning_effort']}, "      f"${GEN['pricing'][0]:.2f}/${GEN['pricing'][1]:.2f} per 1M in/out")

In [ ]:
VAL = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]SRC = [r['input'] for r in VAL]# The Qwen arms are the cross-model reference, not the contrast: this notebook generates# its own baseline, because a selection delta only reads under one fixed generator.QWEN_ROWS = {}for cond in (BASELINE, ARM):    rows = [json.loads(x) for x in (OUT / f'{cond}_{SPLIT}.jsonl').open(encoding='utf-8')            if x.strip()]    assert [r['input'] for r in rows] == SRC, f'{cond} is not aligned to {SPLIT}.jsonl'    QWEN_ROWS[cond] = rowsprint(f'{len(VAL)} {SPLIT} segments; Qwen arms present on {QWEN_ROWS[ARM][0]["model"]}')

---## 3 — The rarity list and the index

In [ ]:
RARITY = json.loads(RARITY_PATH.read_text(encoding='utf-8'))RARITY_SHA = hashlib.sha256(RARITY_PATH.read_bytes()).hexdigest()for key in ('min_df', 'freeze_n', 'zwnj'):    assert RARITY['config'][key] == RAR[key], (key, RARITY['config'][key], RAR[key])assert len(RARITY['terms']) == RAR['freeze_n'], f"{len(RARITY['terms'])} terms on the list"assert RAR['min_df'] <= RARITY['df_observed'][0], RARITY['df_observed']print(f"{RARITY['n_frozen']} frozen terms of {RARITY['n_eligible']} eligible, "      f"realized df {RARITY['df_observed']}, sha256 {RARITY_SHA[:16]}...")

In [ ]:
# data/knn_index is git-ignored, so a fresh session rebuilds it. It must be the index the# Qwen arms retrieved from, hence base_qwen.yaml rather than this config.INDEX = Path(RETR['index_dir'])INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')if not all((INDEX / f).exists() for f in INDEX_FILES):    !{PY} manage.py build_index --config configs/base_qwen.yamlmeta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))assert meta['embed_model'] == RETR['embed_model'], metaprint(f"{meta['n_passages']} pool passages on {meta['embed_model']}, dim {meta['dim']}")

---## 4 — RoutingSelection reads the query and the index, not the generator, so the routes here must be theones the committed diagnostic recorded for the Qwen arm.

In [ ]:
from src.retrieval.rarity import load_irregularfrom src.retrieval.retrieve import RetrievalIndexfrom src.retrieval.sparse import ROUTES, SparseRetrieverindex = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])retriever = SparseRetriever(    index, load_irregular(str(RARITY_PATH)), index, zwnj=RAR['zwnj'], m=SPA['m'],)SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])print(f'{len(TRACES)} traces, {len(SELECTED[0])} exemplars per prompt')

In [ ]:
SLOTS = min(SPA['m'], RETR['k'])HIST = {str(v): sum(t['n_sparse'] == v for t in TRACES) for v in range(SLOTS + 1)}ROUTE_COUNTS = {r: sum(t['route'] == r for t in TRACES) for r in ROUTES}diag = json.loads(DIAG.read_text(encoding='utf-8'))assert diag['config']['index_dir'] == RETR['index_dir'], diag['config']assert HIST == diag['n_sparse']['histogram'], f'{HIST} against {diag["n_sparse"]["histogram"]}'assert ROUTE_COUNTS == diag['routes'], f'{ROUTE_COUNTS} against {diag["routes"]}'ROUTED = len(TRACES) - ROUTE_COUNTS['dense']print(f'routes {ROUTE_COUNTS}  ({ROUTED / len(TRACES):.1%} routed), matches {DIAG}')print(f'rare slots filled {HIST}, mean {np.mean([t["n_sparse"] for t in TRACES]):.3f}')

---## 5 — The prompts the two arms will be sent

In [ ]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, order_exemplarsK = RETR['k']STYLE = Path(PROMPT['style_instruction_file']).read_text(encoding='utf-8')GLOSSARY = _load_configured_glossary(CFG)ORDERED = [order_exemplars(ex, PROMPT['ordering']) for ex in SELECTED]PROMPTS = [build_fewshot_user(s, ex, GLOSSARY) for s, ex in zip(SRC, ORDERED)]BASE_ORDERED = [order_exemplars(ex, PROMPT['ordering']) for ex in index.retrieve(SRC, k=K)]BASE_PROMPTS = [build_fewshot_user(s, ex, GLOSSARY) for s, ex in zip(SRC, BASE_ORDERED)]for i, ex in enumerate(ORDERED):    keys = [(e['input'], e['output']) for e in ex]    assert len(keys) == K, f'segment {i}: {len(keys)} exemplars, expected {K}'    assert len(set(keys)) == len(keys), f'segment {i}: an exemplar repeats across the channels'    assert SRC[i] not in {e['input'] for e in ex}, f'segment {i}: the query is its own exemplar'CHARS = {ARM_NAME: np.array([len(p) for p in PROMPTS]),         BASE_NAME: np.array([len(p) for p in BASE_PROMPTS])}for name, a in CHARS.items():    print(f'{name:20s} median {np.median(a):6.0f}  mean {a.mean():6.0f}  max {a.max():6d} chars')

In [ ]:
i = int(np.flatnonzero(np.array([t['n_sparse'] for t in TRACES]) == SLOTS)[0])print(STYLE)print('=' * 88)print(PROMPTS[i])

---## 6 — Pilot, so the full pass is priced by measurementTwo arms, because the Qwen `knn_fewshot` file cannot serve as the contrast for agpt-5.6-sol arm: it would confound selection with the generator. The pilot rows are thefirst rows of the real pass, so section 7 resumes over them rather than re-buying them.

In [ ]:
N_PILOT = 15PILOT_CAP_USD = 1.00ARMS = [(BASELINE, BASE_NAME), (ARM, ARM_NAME)]# A working ceiling on Persian tokenisation; the pilot replaces it with the model's counts.TOK_PER_CHAR = 0.8in_rate, out_rate = GEN['pricing']guess = sum(    (TOK_PER_CHAR * (CHARS[name].mean() + len(STYLE)) * in_rate + 256 * out_rate) / 1e6    for _cond, name in ARMS) * N_PILOTassert guess <= PILOT_CAP_USD, f'pilot projects to ${guess:.2f} over the ${PILOT_CAP_USD:.2f} cap'print(f'{len(ARMS) * N_PILOT} pilot calls, ${guess:.2f} at the ceiling rate')

In [ ]:
PILOT_CONFIG = Path(tempfile.gettempdir()) / 'gpt56_sparse_knn_pilot.yaml'_pilot = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))_pilot['data']['limit'] = N_PILOTPILOT_CONFIG.write_text(yaml.safe_dump(_pilot, sort_keys=False), encoding='utf-8')PILOT_USAGE = {}for cond, name in ARMS:    r = subprocess.run([PY, 'manage.py', 'infer', '--condition', cond,                        '--config', str(PILOT_CONFIG), '--out-name', name], check=False)    assert r.returncode == 0, f'{name} pilot exited {r.returncode}'    PILOT_USAGE[name] = json.loads(        (OUT / f'{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))

In [ ]:
RATE, PROJECTED = {}, 0.0PILOT_SPENT = 0.0for _cond, name in ARMS:    u = PILOT_USAGE[name]    assert u['calls'] == N_PILOT, f'{name}: {u["calls"]} pilot calls, expected {N_PILOT}'    RATE[name] = u['cost_usd'] / u['calls']    PILOT_SPENT += u['cost_usd']    PROJECTED += RATE[name] * (len(VAL) - N_PILOT)    print(f"{name:20s} {u['prompt_tokens'] / u['calls']:7.0f} in / "          f"{u['completion_tokens'] / u['calls']:5.0f} out per call   ${RATE[name]:.4f} each")print(f'pilot spent ${PILOT_SPENT:.2f}; the remaining '      f'{len(ARMS) * (len(VAL) - N_PILOT)} calls project to ${PROJECTED:.2f}')

In [ ]:
# Left False so a top-to-bottom re-run cannot authorise itself.SPEND_OK = FalseBUDGET_USD = 30.00assert PROJECTED <= BUDGET_USD, f'projection ${PROJECTED:.2f} exceeds ${BUDGET_USD:.2f}'print(f'authorised {SPEND_OK}   cap ${BUDGET_USD:.2f}   projected ${PROJECTED:.2f}')

---## 7 — Generation

In [ ]:
GEN_USAGE = {}t0 = time.perf_counter()for cond, name in ARMS:    assert SPEND_OK, 'set SPEND_OK = True in the cell above to authorise the full pass'    r = subprocess.run([PY, 'manage.py', 'infer', '--condition', cond, '--config', str(CONFIG),                        '--out-name', name], check=False)    assert r.returncode == 0, f'{name} exited {r.returncode}'    GEN_USAGE[name] = json.loads(        (OUT / f'{name}_{SPLIT}_usage.json').read_text(encoding='utf-8'))GEN_SECONDS = round(time.perf_counter() - t0, 1)print(f'{GEN_SECONDS / 60:.1f} min, finished {datetime.now(timezone.utc).isoformat()}')

---## 8 — The outputs, their provenance, and the bill

In [ ]:
ROWS = {}for _cond, name in ARMS:    rows = [json.loads(x) for x in (OUT / f'{name}_{SPLIT}.jsonl').open(encoding='utf-8')            if x.strip()]    assert len(rows) == len(VAL), f'{name}: {len(rows)} rows, expected {len(VAL)}'    assert [r['input'] for r in rows] == SRC, f'{name}: source order differs from the eval file'    assert all(r['model'] == GEN['model'] for r in rows), f'{name}: a different model'    errored = [i for i, r in enumerate(rows) if 'error' in r]    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]    assert not errored, f'{name}: {len(errored)} segments recorded an error: {errored[:5]}'    print(f'{name:20s} {len(rows)} rows, {len(blank)} blank, {len(errored)} errors')    ROWS[name] = rows

In [ ]:
PROV = GEN_USAGE[ARM_NAME]['provenance']assert PROV['rarity_sha256'] == RARITY_SHA, f"{PROV['rarity_sha256']} != {RARITY_SHA}"assert PROV['k'] == RETR['k'] and PROV['m'] == SPA['m'], PROVassert all(PROV[key] == RAR[key] for key in ('min_df', 'freeze_n')), PROVassert PROV['ordering'] == PROMPT['ordering'], PROVassert GEN_USAGE[BASE_NAME]['provenance']['k'] == RETR['k'], 'the arms are not matched on k'print(json.dumps(PROV, indent=2))

In [ ]:
SPENT = PILOT_SPENT + sum(u['cost_usd'] for u in GEN_USAGE.values())CALLS = len(ARMS) * N_PILOT + sum(u['calls'] for u in GEN_USAGE.values())assert SPENT <= BUDGET_USD + PILOT_CAP_USD, f'${SPENT:.2f} spent against the caps'for _cond, name in ARMS:    u = GEN_USAGE[name]    print(f"{name:20s} {u['calls']:5d} resumed calls  ${u['cost_usd']:.2f}")print(f'{CALLS} paid generation calls, ${SPENT:.2f} on {GEN["model"]} this session')

---## 9 — Divergence

In [ ]:
ARM_PRED = [r['prediction'] for r in ROWS[ARM_NAME]]BASE_PRED = [r['prediction'] for r in ROWS[BASE_NAME]]DIFFERS = np.array([a != b for a, b in zip(ARM_PRED, BASE_PRED)])ROUTE = np.array([t['route'] for t in TRACES])share = DIFFERS.mean()print(f'{DIFFERS.sum()}/{len(DIFFERS)} predictions differ ({share:.1%}); '      f'routed fraction is {ROUTED / len(TRACES):.1%}')for r in ROUTES:    sel = ROUTE == r    print(f'  {r:8s} n={sel.sum():5d}  differ {DIFFERS[sel].mean():.1%}')

In [ ]:
dense_diff = DIFFERS[ROUTE == 'dense'].mean()assert share >= 0.40, (    f'only {share:.1%} of predictions differ against a {ROUTED / len(TRACES):.1%} routed '    f'fraction; the rarity channel is not reaching the prompt')if dense_diff:    print(f'NOTE: {dense_diff:.1%} of dense-routed segments differ on an identical prompt. '          f'The API is not deterministic at seed 42, so read the deltas below against this '          f'floor rather than against zero.')else:    print('every dense-routed segment reproduces its pair exactly')

In [ ]:
i = int(np.flatnonzero((ROUTE == 'full') & DIFFERS)[0])print('SOURCE  :', SRC[i][:110])print('terms   :', TRACES[i]['query_terms'][:8], f"served {TRACES[i]['served_terms']}")print(f'{BASE_NAME:20s}:', BASE_PRED[i][:200])print(f'{ARM_NAME:20s}:', ARM_PRED[i][:200])print(f'{ARM + " (qwen)":20s}:', QWEN_ROWS[ARM][i]['prediction'][:200])

---## 10 — Free metrics

In [ ]:
CONDS = [BASE_NAME, ARM_NAME]!{PY} manage.py eval --conditions {' '.join(CONDS + [BASELINE, ARM])} --split {SPLIT}

In [ ]:
from src.eval.quick import scoreSURFACE = {c: score(c, OUT, SPLIT) for c in CONDS + [BASELINE, ARM]}print(f"{'condition':22s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")for cond in CONDS + [BASELINE, ARM]:    s = SURFACE[cond]    print(f"{cond:22s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")print(f"gold targets carry {SURFACE[ARM]['ref_marker_rate']:.2f} markers per segment")

In [ ]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split train

In [ ]:
STYLO_PATH = f'results/stylometrics_ci_{ARM_NAME}_{SPLIT}.json'!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(CONDS)} \    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} --results_path {STYLO_PATH}

In [ ]:
STYLO = json.loads(Path(STYLO_PATH).read_text(encoding='utf-8'))for cond in CONDS:    print(f"  {cond:22s} stylo_dist {STYLO['cells'][cond]['stylo_dist']:.4f}")

### COMET

In [ ]:
if not Path(COMET_PY).exists():    pip = [COMET_PY, '-m', 'pip', 'install', '-q']    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)    subprocess.run([*pip, '--upgrade', 'pip'], check=True)    subprocess.run([*pip, 'setuptools<81'], check=True)    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)COMET_PATH = f'results/comet_{SPLIT}.json'PRIOR_COMET = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))r = subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', *CONDS, '--split', SPLIT,                    '--results_path', COMET_PATH, '--batch_size', '16'], check=False)assert r.returncode == 0, f'comet exited {r.returncode}'

In [ ]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))assert PRIOR_COMET <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR_COMET - set(COMET))}'for cond in CONDS + [BASELINE, ARM]:    if cond in COMET:        assert COMET[cond]['sources'] == COMET[BASE_NAME]['sources'], f'{cond} is not paired'        print(f"  {cond:22s} COMET {COMET[cond]['system']:.4f}")

---## 11 — Judge Φ (paid)The primary rater is `claude-haiku-4-5`, cross-family from the generator. The second raterin `configs/judge_eval_gpt.yaml` is a gpt-5.6 model, so it would be scoring its own familyhere; it is left out rather than reported as an independent check.

In [ ]:
JUDGE_CFG = 'configs/judge_eval.yaml'JUDGE_RESULTS = f'results/judge_{SPLIT}.json'JUDGE_USAGE = f'results/judge_{SPLIT}_usage.json'JUDGE_CI_PATH = f'results/judge_ci_{ARM_NAME}_{SPLIT}.json'PRIOR_JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))PRIOR_USAGE = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))PRIOR_SPEND, PRIOR_CALLS = (PRIOR_USAGE['cumulative'][k] for k in ('cost_usd', 'calls'))BUY = [c for c in CONDS if c not in PRIOR_JUDGE]PER_CALL = PRIOR_SPEND / PRIOR_CALLSN_CALLS = len(VAL) * len(BUY)PROJECTED_J = PER_CALL * N_CALLSprint(f"buying Phi for {BUY or 'nothing'}: {N_CALLS} calls at ${PER_CALL:.5f} "      f"= ${PROJECTED_J:.2f} projected (cumulative judge spend ${PRIOR_SPEND:.2f})")

In [ ]:
if not os.environ.get('ANTHROPIC_API_KEY'):    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY: ')# Left False so a top-to-bottom re-run cannot authorise itself.SPEND_OK_J = FalseBUDGET_J_USD = 3.00assert PROJECTED_J <= BUDGET_J_USD, f'projection ${PROJECTED_J:.2f} over ${BUDGET_J_USD:.2f}'print(f'authorised {SPEND_OK_J}   cap ${BUDGET_J_USD:.2f}   projected ${PROJECTED_J:.2f}')

In [ ]:
# Re-running over a complete cache makes no request.if not BUY:    print('nothing to buy: both arms already carry Phi')else:    assert SPEND_OK_J, 'set SPEND_OK_J = True in the cell above to authorise the pass'    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,                        '--config', JUDGE_CFG], check=False)    assert r.returncode == 0, f'judge exited {r.returncode}'

In [ ]:
JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))HAVE_PHI = all(c in JUDGE for c in CONDS)lost = sorted(set(PRIOR_JUDGE) - set(JUDGE))assert not lost, f'lost from {JUDGE_RESULTS}: {lost}'usage = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))assert usage['priced'], 'the judge model has no pricing table; cost_usd is a floor, not a bill'SPENT_J = usage['cumulative']['cost_usd'] - PRIOR_SPENDassert SPENT_J <= BUDGET_J_USD, f'${SPENT_J:.2f} spent against a ${BUDGET_J_USD:.2f} cap'if HAVE_PHI:    for cond in CONDS:        assert JUDGE[cond]['model'] == JUDGE[BASE_NAME]['model'], f'{cond}: two raters, not one'    r = subprocess.run([PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *CONDS,                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),                        '--results_path', JUDGE_CI_PATH], check=False)    assert r.returncode == 0, f'judge_ci exited {r.returncode}'METRICS = ['chrf', 'bleu', 'comet'] + (['judge'] if HAVE_PHI else [])print(f"{usage['cumulative']['calls'] - PRIOR_CALLS} paid calls, ${SPENT_J:.2f} on "      f"{usage['model']}; reading out on {', '.join(METRICS)}")

---## 12 — The paired bootstrap

In [ ]:
BOOT_PATHS = {}for metric in METRICS:    path = f'results/bootstrap_{metric}_{ARM_NAME}_{SPLIT}.json'    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric, '--conditions', *CONDS,                        '--split', SPLIT, '--pairs', f'{ARM_NAME}:{BASE_NAME}',                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),                        '--out', path], check=False)    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'    BOOT_PATHS[metric] = path

---## 13 — Read-out by doseThe floors are the primary rater's, declared at n=1,323 and scaled by 1/sqrt(n) for thesubsets. A non-deterministic API adds decode noise on top of them; section 9 measures it.

In [ ]:
from src.eval.bootstrap import _load_segment_scores, paired_bootstrapSTRATA = {    f'full dose (n_sparse = {SLOTS})': [i for i, t in enumerate(TRACES) if t['n_sparse'] == SLOTS],    'all routed': [i for i, t in enumerate(TRACES) if t['route'] != 'dense'],    'all segments': list(range(len(VAL))),}FLOOR = {'judge': 0.058, 'comet': 0.005}print({k: len(v) for k, v in STRATA.items()})

In [ ]:
SCORES = {}for metric in METRICS:    scores, sources = _load_segment_scores(metric, CONDS, OUT, SPLIT, None)    for cond in CONDS:        assert len(scores[cond]) == len(VAL), (metric, cond, len(scores[cond]))        if sources.get(cond) is not None:            assert sources[cond] == SRC, f'{metric}/{cond}: segment order is not the eval order'    SCORES[metric] = scoresprint('per-segment scores aligned to the eval order for', ', '.join(SCORES))

In [ ]:
PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4, 'judge': 4}for metric in METRICS:    p = PLACES[metric]    print(f'\n{metric}   {ARM_NAME} - {BASE_NAME}')    for label, idx in STRATA.items():        d = paired_bootstrap([SCORES[metric][ARM_NAME][i] for i in idx],                             [SCORES[metric][BASE_NAME][i] for i in idx],                             n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)        line = (f"  {label:26s} n={d['n']:5d}  {d['diff']:+.{p}f} "                f"[{d['ci_low']:+.{p}f}, {d['ci_high']:+.{p}f}]  p={d['p_value']:.4f} "                f"{'*' if d['significant'] else ' '}")        if metric in FLOOR:            f = FLOOR[metric] * (len(VAL) / d['n']) ** 0.5            line += f"  floor {f:.{p}f}{'' if abs(d['diff']) >= f else '  (under)'}"        print(line)

In [ ]:
# The same contrast under Qwen, so the two generators can be read side by side.for metric in METRICS:    p = PLACES[metric]    q, _ = _load_segment_scores(metric, [BASELINE, ARM], OUT, SPLIT, None)    if not all(c in q for c in (BASELINE, ARM)):        continue    idx = STRATA['all segments']    dq = paired_bootstrap([q[ARM][i] for i in idx], [q[BASELINE][i] for i in idx],                          n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)    dg = paired_bootstrap([SCORES[metric][ARM_NAME][i] for i in idx],                          [SCORES[metric][BASE_NAME][i] for i in idx],                          n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)    print(f"{metric:6s}  qwen {dq['diff']:+.{p}f} [{dq['ci_low']:+.{p}f}, {dq['ci_high']:+.{p}f}]"          f"   {GEN['model']} {dg['diff']:+.{p}f} "          f"[{dg['ci_low']:+.{p}f}, {dg['ci_high']:+.{p}f}]")